# 🚀 Быстрый старт: Использование NLP SDK

Этот ноутбук демонстрирует, как использовать класс `TextClassifier` для быстрого запуска модели машинного обучения в 3 строчки кода. Фасад автоматически подтягивает конфигурацию, настраивает устройство (CPU/GPU) и загружает веса.

В этом примере мы тестируем модель на задаче бинарной классификации (например, определение спама).

In [1]:
# Настройка логирования для красивого вывода в ноутбуке
import logging
import sys
from pathlib import Path


logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

# Настраиваем пути, чтобы ноутбук видел модули из папки src
project_root = str(Path.cwd().parent)
if project_root not in sys.path:
    sys.path.append(project_root)

## Инициализация классификатора

При инициализации SDK:
1. Автоматически парсится `configs/main.yaml`
2. Загружается токенизатор и базовая модель (`HFModelBuilder`)
3. Переносится на GPU (если доступно)

*Опционально:* Если у вас есть обученные веса, вы можете передать параметры `checkpoint_path` и `hf_repo_id` для автоматического скачивания.

In [2]:
from src.sdk.inference import NLPPipeline


# Инициализация классификатора
# Если вы уже обучили модель и залили на HF:
# classifier = NLPPipeline(hf_repo_id="your_name/fake-news-model", checkpoint_path="best.ckpt")

classifier = NLPPipeline()

# Тестовый прогон
result = classifier("Власти опровергли слухи о закрытии станций метро.")
print(result)

c:\fake-news-detection-ml-system\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO: Инициализация NLPPipeline (Classification) на устройстве: cpu
INFO: Загрузка токенизатора: DeepPavlov/rubert-base-cased
INFO: Загрузка базовой модели: DeepPavlov/rubert-base-cased
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 16582.56it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias          

[{'label_id': 1, 'confidence': 0.5211, 'all_probabilities': [0.4789, 0.5211]}]


## Запуск Инференса

Мы можем передавать как одиночные строки, так и батчи (списки строк). Метод `predict` возвращает структурированный словарь с вероятностями и предсказанным классом.

In [3]:
# Тестовые данные (имитация входящих новостей)
sample_messages = [
    "Здравствуйте, подскажите статус моей заявки №12345?",
    "ШОК! Инопланетяне высадились в Саратове! Смотреть без смс...",
    "Центробанк сохранил ключевую ставку на уровне 16%."
]

# Инференс (предполагаем, что NLPPipeline поддерживает вызов через __call__ или predict)
results = classifier(sample_messages)

# Красивый вывод результатов (добавлен strict=True для Python 3.10+)
for msg, res in zip(sample_messages, results, strict=True):
    print("-" * 50)
    print(f"Текст: {msg}")
    print(f"Класс (ID): {res['label_id']}")
    print(f"Уверенность: {res['confidence'] * 100:.2f}%")

--------------------------------------------------
Текст: Здравствуйте, подскажите статус моей заявки №12345?
Класс (ID): 1
Уверенность: 50.27%
--------------------------------------------------
Текст: ШОК! Инопланетяне высадились в Саратове! Смотреть без смс...
Класс (ID): 1
Уверенность: 54.25%
--------------------------------------------------
Текст: Центробанк сохранил ключевую ставку на уровне 16%.
Класс (ID): 1
Уверенность: 50.14%
